# Zlab Cloud · Ball approach stability
Use a Kaggle GPU runtime with Internet access. Attach checkpoint 1400 as a Kaggle input or point the training cell to its Hugging Face path.
The nine-hour training budget runs 100-update blocks at a fixed 2.8 m/s. The notebook saves the best evaluated checkpoint and only increases disturbances after passing the stability checks.

In [ ]:
import os
from pathlib import Path
import subprocess
import tempfile

repo = Path(tempfile.mkdtemp(prefix="zlab-cloud-")) / "Zlab-cloud"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/hafizadha/Zlab-cloud.git", str(repo),
], check=True)
project = repo / "Zlab_Gym"
launcher = project / "train_ball_stability.sh"
if not launcher.is_file():
    raise RuntimeError("This checkout does not contain the ball stability launcher.")
subprocess.run(["git", "-C", str(repo), "log", "-1", "--oneline"], check=True)


import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[huggingface]", "onnx"], cwd=project, check=True)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator before training.")

In [ ]:
# Set this to False to keep checkpoints only in Kaggle.
# For uploads, add a write token in Kaggle Secrets.
UPLOAD_TO_HF = True

if UPLOAD_TO_HF:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["ZLAB_HF_REPO_ID"] = "hafiz24/zlab-checkpoints"


In [ ]:
HOURS = 9.0
UPDATES_PER_BLOCK = 100
ENVS_PER_GPU = 128
FOOT_FRICTION_RANGE = (0.35, 1.15)  # Broad simulation range until you measure your turf.
START_CHECKPOINT = ""  # Absolute path to model_1400.pt or a saved stability checkpoint.
HF_START_FILE = ""  # e.g. checkpoints/<run>/model_1400.pt
WORK_DIR = Path("/kaggle/working/zlab-ball-stability")
progress = WORK_DIR / "progress.json"

if START_CHECKPOINT and HF_START_FILE:
    raise ValueError("Choose one checkpoint source.")
if progress.is_file():
    import json
    checkpoint = Path(json.loads(progress.read_text())["checkpoint"])
elif START_CHECKPOINT:
    checkpoint = Path(START_CHECKPOINT)
elif HF_START_FILE:
    from huggingface_hub import hf_hub_download

    checkpoint = Path(hf_hub_download(
        repo_id="hafiz24/zlab-checkpoints", filename=HF_START_FILE, repo_type="model"
    ))
else:
    matches = list(Path("/kaggle/input").rglob("model_1400.pt"))
    if len(matches) != 1:
        raise ValueError("Attach model_1400.pt or set START_CHECKPOINT or HF_START_FILE.")
    checkpoint = matches[0]

subprocess.run([
    sys.executable, "scripts/train_stability_blocks.py", str(checkpoint),
    "--work-dir", str(WORK_DIR), "--hours", str(HOURS),
    "--updates", str(UPDATES_PER_BLOCK), "--num-envs", str(ENVS_PER_GPU),
], cwd=project, env=dict(
    os.environ, ZLAB_USE_ALL_GPUS="1",
    ZLAB_FOOT_FRICTION_MIN=str(FOOT_FRICTION_RANGE[0]),
    ZLAB_FOOT_FRICTION_MAX=str(FOOT_FRICTION_RANGE[1]),
), check=True)

In [ ]:
# Independently evaluate the selected policy after training.
import json

if progress.is_file():
    selected = json.loads(progress.read_text())["checkpoint"]
    subprocess.run([
        sys.executable, "scripts/play.py",
        "--checkpoint-file", selected,
        "--device", "cuda:0", "--eval-episodes", "96", "--num-envs", "48",
        "--seed", "3067", "--disturbances", "True", "--torso-push-scale", "0.25",
        "--observation-noise-scale", "0.25",
        "--foot-friction-min", str(FOOT_FRICTION_RANGE[0]),
        "--foot-friction-max", str(FOOT_FRICTION_RANGE[1]),
    ], cwd=project, check=True)